In [25]:
# 1. Imports and configuration — edit paths here, then run all cells.
import re
import json
import time
import shutil
import zipfile
import hashlib
import platform
import warnings
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import sklearn
import joblib
from threadpoolctl import threadpool_limits
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, 'to_string') else value)

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

DATASET_ZIP = None  # Unused here: Kaggle has already extracted the dataset.
RUN_MODE = 'full'  # 'full' = all six primary feature sets and all declared hyperparameters.
USE_GOOGLE_DRIVE = False
OUTPUT_DIRECTORY = None  # Optional absolute path; otherwise chosen below.
RESUME = True
TRAIN_FINAL_MODEL = True
SEED = 42
THREAD_LIMIT = 1  # Small datasets often run faster without BLAS oversubscription.
PIPELINE_VERSION = 'smartglove-corrected-1.1-eight-fixes'

WIN = 50
SAMPLE_PERIOD_MS = 40
ACC_LSB_PER_G = 8192.0
GYRO_LSB_PER_DPS = 16.4
SPAN_ZERO_EPS = 1e-12
FLEX = ['flex_thumb', 'flex_index', 'flex_middle', 'flex_ring', 'flex_little']
ACC = ['ax', 'ay', 'az']
GYRO = ['gx', 'gy', 'gz']
QUAT = ['qw', 'qx', 'qy', 'qz']
RAW_SENSOR_COLS = FLEX + ACC + GYRO + QUAT
TRIAL_KEYS = ['participant', 'gesture', 'trial_id']
CAL_KEYS = ['participant', 'session', 'ref_id']
META_COLS = TRIAL_KEYS + ['session', 'ref_id', 'round', 'position']

FEATURE_SET_ORDER = [
    'F1_RAW_FLEX',
    'F2_NEUTRAL_FLEX',
    'F3_RAW_FLEX_GRAVITY',
    'F4_NEUTRAL_FLEX_GRAVITY',
    'F5_NEUTRAL_FLEX_STATIC_IMU',
    'F6_NEUTRAL_FLEX_IQR',
]
MODE_LABELS = {
    'F1_RAW_FLEX': 'F1 Raw Flex',
    'F2_NEUTRAL_FLEX': 'F2 Neutral Flex',
    'F3_RAW_FLEX_GRAVITY': 'F3 Raw Flex + Relative Gravity',
    'F4_NEUTRAL_FLEX_GRAVITY': 'F4 Neutral Flex + Relative Gravity',
    'F5_NEUTRAL_FLEX_STATIC_IMU': 'F5 Neutral Flex + Static IMU',
    'F6_NEUTRAL_FLEX_IQR': 'F6 Neutral Flex + Flex IQR',
}

if RUN_MODE not in {'full', 'smoke'}:
    raise ValueError("RUN_MODE must be 'full' or 'smoke'.")
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
default_output = (Path('/content/drive/MyDrive/SmartGlove_Final')
                  if USE_GOOGLE_DRIVE else Path.cwd() / 'smartglove_corrected_results')
OUTPUT_BASE = Path(OUTPUT_DIRECTORY).expanduser() if OUTPUT_DIRECTORY else default_output
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scipy': scipy.__version__,
            'scikit_learn': sklearn.__version__, 'joblib': joblib.__version__}
print('Versions:', VERSIONS)
print('Run mode:', RUN_MODE)
print('Output base:', OUTPUT_BASE.resolve())
if RUN_MODE == 'smoke':
    print('SOFTWARE CHECK ONLY: smoke results are not the full experiment.')


Versions: {'python': '3.12.13', 'numpy': '2.0.2', 'pandas': '2.3.3', 'scipy': '1.16.3', 'scikit_learn': '1.6.1', 'joblib': '1.5.3'}
Run mode: full
Output base: /kaggle/working/smartglove_corrected_results


## 2. Load the Kaggle dataset

Kaggle has already extracted the uploaded dataset. This section uses the mounted
`SmartGlove_Dataset` directory directly, verifies its structure, computes a deterministic
SHA-256 fingerprint of the dataset directory for reproducibility, and loads the gesture map.


In [26]:
from pathlib import Path

# ============================================================
# Kaggle dataset location
# ============================================================

DATA_ROOT = Path(
    "/kaggle/input/datasets/ratuldas07/smartglove/SmartGlove_Dataset"
)

# Verify the expected dataset structure.
required_items = [
    DATA_ROOT / "participants.csv",
    DATA_ROOT / "gesture_map.csv",
    DATA_ROOT / "gesture_definitions.csv",
    DATA_ROOT / "raw",
]

missing = [str(p) for p in required_items if not p.exists()]
if missing:
    raise FileNotFoundError(
        "SmartGlove dataset is incomplete or the Kaggle path is wrong.\n"
        "Missing:\n" + "\n".join(missing)
    )

# Deterministic SHA-256 fingerprint for the already-extracted Kaggle dataset.
# This replaces the old ZIP hash so later checkpoint/metadata code still works.
def sha256_directory(root):
    root = Path(root)
    h = hashlib.sha256()
    files = sorted(p for p in root.rglob("*") if p.is_file())
    for path in files:
        rel = path.relative_to(root).as_posix().encode("utf-8")
        h.update(rel)
        h.update(b"\0")
        with path.open("rb") as f:
            for block in iter(lambda: f.read(1024 * 1024), b""):
                h.update(block)
    return h.hexdigest()

DATASET_SHA256 = sha256_directory(DATA_ROOT)

# Load gesture definitions required by later cells.
gesture_map = pd.read_csv(
    DATA_ROOT / "gesture_map.csv",
    dtype={"gesture_id": str}
)

if gesture_map["gesture_id"].duplicated().any():
    raise ValueError("Duplicate gesture IDs in gesture_map.csv.")

LABELS = sorted(gesture_map["gesture_id"].tolist())
gesture_names = dict(
    zip(gesture_map["gesture_id"], gesture_map["gesture_name"])
)

print("Dataset root:", DATA_ROOT)
print("Dataset SHA-256:", DATASET_SHA256)
print("Gestures:", LABELS)
print("Dataset found successfully.")


Dataset root: /kaggle/input/datasets/ratuldas07/smartglove/SmartGlove_Dataset
Dataset found successfully.


## 3. Reconstruct the accepted trials

The acquisition log determines inclusion. Rejected/discarded attempts are not
converted into additional training observations. Technical checks fail explicitly
on missing files, duplicate trial IDs, malformed timestamps or non-finite inputs;
they do not remove rows or use classification performance.


In [ ]:
def load_accepted_trials(data_root):
    sample_parts, meta_rows = [], []

    participant_dirs = sorted(
        p for p in (data_root / "raw").iterdir()
        if p.is_dir() and re.fullmatch(r"P\d+", p.name)
    )

    for pdir in participant_dirs:
        pid = pdir.name

        log = pd.read_csv(pdir / "trial_log.csv")
        needed = {
            "result", "data_file", "gesture_id", "gesture_name", "trial_id",
            "session", "ref_id", "round", "position", "time"
        }

        if not needed.issubset(log.columns):
            raise ValueError(
                f"{pid}: missing trial-log columns {needed - set(log.columns)}"
            )

        accepted = log.loc[log["result"].eq("ACCEPTED")].copy()
        file_cache = {}

        for _, row in accepted.iterrows():
            if row[["session", "ref_id", "gesture_id", "trial_id"]].isna().any():
                raise ValueError(f"{pid}: incomplete accepted-trial identity.")

            file_path = (pdir / str(row["data_file"])).resolve()

            if (
                not file_path.is_relative_to(pdir.resolve())
                or not file_path.is_file()
            ):
                raise FileNotFoundError(f"Invalid trial file: {file_path}")

            if file_path not in file_cache:
                file_cache[file_path] = pd.read_csv(file_path)

            file_df = file_cache[file_path]
            trial = file_df.loc[file_df["trial_id"].eq(row["trial_id"])].copy()

            if len(trial) != WIN:
                raise ValueError(
                    f"{pid} {row.gesture_id} trial {row.trial_id}: "
                    f"{len(trial)} rows, expected {WIN}."
                )

            required = RAW_SENSOR_COLS + ["timestamp_ms"]

            if not set(required).issubset(trial.columns):
                raise ValueError(
                    f"{file_path}: missing sensor/timestamp columns."
                )

            values = trial[required].to_numpy(dtype=float)

            if not np.isfinite(values).all():
                raise ValueError(
                    f"{pid} {row.gesture_id}: non-finite sensor/timestamp value."
                )

            if not np.all(
                np.diff(trial["timestamp_ms"].to_numpy(float))
                == SAMPLE_PERIOD_MS
            ):
                raise ValueError(
                    f"{pid} {row.gesture_id}: sampling intervals differ from 40 ms. "
                    "Check acquisition settings; no samples were removed."
                )

            identity = {
                "participant": pid,
                "gesture": str(row["gesture_id"]),
                "session": str(row["session"]),
                "ref_id": str(row["ref_id"]),
                "round": row["round"],
                "position": row["position"],
            }

            for offset, (name, value) in enumerate(identity.items()):
                trial.insert(offset, name, value)

            sample_parts.append(trial)

            meta_rows.append({
                **identity,
                "gesture_name": row["gesture_name"],
                "trial_id": int(row["trial_id"]),
                "data_file": row["data_file"],
                "trial_time": row["time"],
            })

    if not sample_parts:
        raise ValueError("No ACCEPTED trials were found.")

    samples = pd.concat(sample_parts, ignore_index=True)

    meta = (
        pd.DataFrame(meta_rows)
        .sort_values(TRIAL_KEYS)
        .reset_index(drop=True)
    )

    if meta[TRIAL_KEYS].duplicated().any():
        raise ValueError(
            "Duplicate accepted trial identities. Do not merge separate trials."
        )

    if set(meta["gesture"]) != set(LABELS):
        raise ValueError("Gesture labels differ from gesture_map.csv.")

    for pid, rows in meta.groupby("participant"):
        if set(rows["gesture"]) != set(LABELS):
            raise ValueError(
                f"{pid}: every gesture must be represented for this protocol."
            )

    # Locked final-dataset integrity checks.
    expected_participants = [f"P{i:02d}" for i in range(1, 11)]
    expected_gestures = [f"G{i:02d}" for i in range(1, 11)]

    if sorted(meta["participant"].unique()) != expected_participants:
        raise ValueError(
            f"Expected participants {expected_participants}, "
            f"found {sorted(meta['participant'].unique())}."
        )

    if sorted(LABELS) != expected_gestures:
        raise ValueError(
            f"Expected gestures {expected_gestures}, found {sorted(LABELS)}."
        )

    if len(meta) != 500:
        raise ValueError(
            f"Expected exactly 500 accepted trials, found {len(meta)}."
        )

    if len(samples) != 25000:
        raise ValueError(
            f"Expected exactly 25,000 accepted sensor rows, found {len(samples)}."
        )

    per_person = meta.groupby("participant").size()
    if not (per_person == 50).all():
        raise ValueError(
            f"Expected 50 accepted trials per participant:\n{per_person}"
        )

    per_gesture = meta.groupby("gesture").size()
    if not (per_gesture == 50).all():
        raise ValueError(
            f"Expected 50 accepted trials per gesture:\n{per_gesture}"
        )

    per_cell = meta.groupby(["participant", "gesture"]).size()
    if not (per_cell == 5).all():
        raise ValueError(
            f"Expected 5 accepted trials per participant×gesture cell:\n{per_cell}"
        )

    return samples, meta


samples, trial_meta = load_accepted_trials(DATA_ROOT)
participants = sorted(trial_meta["participant"].unique())

print(
    f"{len(participants)} participants | "
    f"{len(LABELS)} gestures | "
    f"{len(trial_meta)} trials | "
    f"{len(samples)} sensor rows"
)

display(
    pd.crosstab(
        trial_meta["participant"],
        trial_meta["gesture"]
    )
)


## 4. Match calibration by participant, session and reference ID

Session calibration is matched exactly by `participant + session + ref_id` and must
precede the matched accepted trials.

Primary F1-F6 modeling uses the session neutral flex means, neutral acceleration
reference, and gyro bias where required. Neutral/fist recordings are never added as
gesture-training examples and references are never averaged across sessions.

The neutral-to-fist span is still computed and exported as a **diagnostic only** so
the original calibration information is preserved. It is not used in the primary
feature-set comparison, consistent with the P07 thumb-reference concern.


In [27]:
def build_calibration_table(data_root, accepted_meta):
    rows, diagnostics = [], []
    for pid, participant_trials in accepted_meta.groupby('participant', sort=True):
        pdir = data_root / 'raw' / pid
        summary = pd.read_csv(pdir / 'calibration_summary.csv')
        events = pd.read_csv(pdir / 'session_log.csv')
        if summary[['session', 'ref_id']].duplicated().any():
            raise ValueError(f'{pid}: duplicate calibration references.')
        for session, ref_id in participant_trials[['session', 'ref_id']].drop_duplicates().itertuples(index=False, name=None):
            hit = summary.loc[summary['session'].eq(session) & summary['ref_id'].eq(ref_id)]
            if len(hit) != 1:
                raise ValueError(f'Missing/ambiguous calibration: {pid}, {session}, {ref_id}.')
            # The fist filename is session-specific, not reference-specific.
            if summary.loc[summary['session'].eq(session), 'ref_id'].nunique() != 1:
                raise ValueError(f'{pid} {session}: multiple neutral references in one session. '
                                 'An explicit reference-to-fist mapping is required.')
            s = hit.iloc[0]
            neutral_samples = pd.read_csv(pdir / f'calibration_{session}.csv')
            fist_samples = pd.read_csv(pdir / f'fist_{session}.csv')
            for tag, frame in [('neutral', neutral_samples), ('fist', fist_samples)]:
                if len(frame) < 2 or not np.isfinite(frame[FLEX].to_numpy(float)).all():
                    raise ValueError(f'{pid} {session}: invalid {tag} calibration samples.')
            neutral = s[[f'{f}_mean' for f in FLEX]].to_numpy(dtype=float)
            if not np.isfinite(neutral).all():
                raise ValueError(f'{pid} {session}: non-finite neutral summary.')
            if not np.allclose(neutral, neutral_samples[FLEX].mean().to_numpy(float), atol=0.01, rtol=0):
                raise ValueError(f'{pid} {session}: neutral summary does not match its sample file.')

            # Span values are retained only as diagnostics; span normalization is not
            # part of the primary F1-F6 experiment.
            fist = fist_samples[FLEX].mean().to_numpy(float)
            span = fist - neutral
            if not np.isfinite(span).all() or (np.abs(span) <= SPAN_ZERO_EPS).any():
                raise ValueError(f'{pid} {session}: non-finite or zero calibration span.')

            gyro_bias = s[['gyro_bias_x_dps', 'gyro_bias_y_dps', 'gyro_bias_z_dps']].to_numpy(dtype=float)
            neutral_acc = s[['accel_x_mean_g', 'accel_y_mean_g', 'accel_z_mean_g']].to_numpy(dtype=float)
            if not np.isfinite(gyro_bias).all():
                raise ValueError(f'{pid} {session}: non-finite gyro-bias reference.')
            if not np.isfinite(neutral_acc).all() or np.linalg.norm(neutral_acc) <= 0:
                raise ValueError(f'{pid} {session}: invalid neutral acceleration reference.')

            matched = participant_trials.loc[participant_trials['session'].eq(session)
                                             & participant_trials['ref_id'].eq(ref_id)]
            first_trial = pd.to_datetime(matched['trial_time']).min()
            fist_events = events.loc[events['session'].eq(session) & events['event'].eq('FIST_REFERENCE')]
            if len(fist_events) != 1:
                raise ValueError(f'{pid} {session}: ambiguous fist-reference event history.')
            if not (pd.Timestamp(s['time']) < first_trial
                    and pd.Timestamp(fist_events.iloc[0]['time']) < first_trial):
                raise ValueError(f'{pid} {session}: calibration was not logged before its trials.')

            row = {'participant': pid, 'session': session, 'ref_id': ref_id,
                   'gyro_bias_x_dps': float(gyro_bias[0]),
                   'gyro_bias_y_dps': float(gyro_bias[1]),
                   'gyro_bias_z_dps': float(gyro_bias[2]),
                   'neutral_accel_x_g': float(neutral_acc[0]),
                   'neutral_accel_y_g': float(neutral_acc[1]),
                   'neutral_accel_z_g': float(neutral_acc[2])}
            for i, f in enumerate(FLEX):
                row[f'{f}_neutral'] = float(neutral[i])
                row[f'{f}_span'] = float(span[i])
                diagnostics.append({**{key: row[key] for key in CAL_KEYS}, 'finger': f,
                                    'neutral_mean_adc': float(neutral[i]),
                                    'fist_mean_adc': float(fist[i]),
                                    'signed_span_adc': float(span[i]),
                                    'absolute_span_adc': float(abs(span[i])),
                                    'neutral_std_adc': float(neutral_samples[f].std(ddof=1)),
                                    'fist_std_adc': float(fist_samples[f].std(ddof=1)),
                                    'matched_trials': len(matched)})
            rows.append(row)
    calibration = pd.DataFrame(rows)
    if calibration[CAL_KEYS].duplicated().any():
        raise ValueError('Calibration keys must be unique.')
    return calibration, pd.DataFrame(diagnostics)

calibration_used, calibration_diagnostics = build_calibration_table(DATA_ROOT, trial_meta)
print('Used calibration references:', len(calibration_used))
display(calibration_used[CAL_KEYS])
print('Smallest observed spans (descriptive only; span normalization is secondary, not primary):')
display(calibration_diagnostics.sort_values('absolute_span_adc').head(5))


NameError: name 'trial_meta' is not defined

## 5. Build the six locked primary feature sets

One feature vector is built per complete 50-sample trial.

- **F1 Raw Flex:** 5 raw flex medians.
- **F2 Neutral Flex:** 5 session-neutral-centered flex medians.
- **F3 Raw Flex + Relative Gravity:** F1 + 3 components of `u - u0`.
- **F4 Neutral Flex + Relative Gravity:** F2 + 3 components of `u - u0`.
- **F5 Neutral Flex + Static IMU:** F4 + acceleration-magnitude median/SD +
  bias-corrected gyro-magnitude median/p95.
- **F6 Neutral Flex + Flex IQR:** F2 + 5 flex IQRs.

Flex medians use the 50 trial samples. IQR uses linear percentile interpolation.
For relative gravity, `u` is the normalized componentwise-median trial acceleration
vector and `u0` is the normalized matching session-neutral acceleration vector.

Gyro values are converted using 16.4 counts/(degree/s), the matching session bias is
subtracted axis-by-axis, and gyro magnitude is computed afterward.

No ANOVA `SelectKBest` step is used in the primary experiment. The exact predictor
block is determined by the chosen F1-F6 feature set inside inner CV.


In [ ]:
RAW_FLEX_MEDIAN_COLS = [f'{f}_median' for f in FLEX]
NEUTRAL_FLEX_MEDIAN_COLS = [f'{f}_centered_median' for f in FLEX]
FLEX_IQR_COLS = [f'{f}_iqr' for f in FLEX]
REL_GRAVITY_COLS = ['gravity_rel_x', 'gravity_rel_y', 'gravity_rel_z']
STATIC_IMU_COLS = ['acc_mag_median_g', 'acc_mag_sd_g',
                   'gyro_mag_median_dps', 'gyro_mag_p95_dps']

FEATURE_SETS = {
    'F1_RAW_FLEX': RAW_FLEX_MEDIAN_COLS,
    'F2_NEUTRAL_FLEX': NEUTRAL_FLEX_MEDIAN_COLS,
    'F3_RAW_FLEX_GRAVITY': RAW_FLEX_MEDIAN_COLS + REL_GRAVITY_COLS,
    'F4_NEUTRAL_FLEX_GRAVITY': NEUTRAL_FLEX_MEDIAN_COLS + REL_GRAVITY_COLS,
    'F5_NEUTRAL_FLEX_STATIC_IMU': NEUTRAL_FLEX_MEDIAN_COLS + REL_GRAVITY_COLS + STATIC_IMU_COLS,
    'F6_NEUTRAL_FLEX_IQR': NEUTRAL_FLEX_MEDIAN_COLS + FLEX_IQR_COLS,
}
if list(FEATURE_SETS) != FEATURE_SET_ORDER:
    raise AssertionError('Feature-set order differs from the declared F1-F6 order.')

def extract_feature_pool(trial, calibration):
    """Compute only the quantities needed by the six locked primary feature sets."""
    if len(trial) != WIN:
        raise ValueError(f'Expected {WIN} samples, got {len(trial)}.')
    if not np.isfinite(trial[RAW_SENSOR_COLS].to_numpy(float)).all():
        raise ValueError('Non-finite sensor input to feature extraction.')
    timestamps = trial['timestamp_ms'].to_numpy(float)
    if not np.isfinite(timestamps).all() or not np.all(np.diff(timestamps) == SAMPLE_PERIOD_MS):
        raise ValueError('Trial timestamps must have 40 ms spacing.')

    out = {}

    # Fix 3: trial-level flex central tendency is the median, not the mean.
    flex = trial[FLEX].to_numpy(float)
    flex_median = np.median(flex, axis=0)
    q25 = np.percentile(flex, 25, axis=0, method='linear')
    q75 = np.percentile(flex, 75, axis=0, method='linear')
    flex_iqr = q75 - q25
    neutral_flex = np.array([float(calibration[f'{f}_neutral']) for f in FLEX])
    centered_median = flex_median - neutral_flex

    for i, f in enumerate(FLEX):
        out[f'{f}_median'] = float(flex_median[i])
        out[f'{f}_centered_median'] = float(centered_median[i])
        out[f'{f}_iqr'] = float(flex_iqr[i])

    # Fix 5: gravity direction is expressed relative to the matching session neutral.
    acc = trial[ACC].to_numpy(float) / ACC_LSB_PER_G
    a_med = np.median(acc, axis=0)
    a_norm = float(np.linalg.norm(a_med))
    if a_norm <= 0:
        raise ValueError('Trial median acceleration vector has zero norm.')
    u = a_med / a_norm

    u0 = np.array([float(calibration['neutral_accel_x_g']),
                   float(calibration['neutral_accel_y_g']),
                   float(calibration['neutral_accel_z_g'])])
    u0_norm = float(np.linalg.norm(u0))
    if u0_norm <= 0:
        raise ValueError('Neutral acceleration vector has zero norm.')
    u0 = u0 / u0_norm
    u_rel = u - u0
    for name, value in zip(REL_GRAVITY_COLS, u_rel):
        out[name] = float(value)

    acc_mag = np.linalg.norm(acc, axis=1)
    out['acc_mag_median_g'] = float(np.median(acc_mag))
    out['acc_mag_sd_g'] = float(acc_mag.std(ddof=1))

    gyro = trial[GYRO].to_numpy(float) / GYRO_LSB_PER_DPS
    gyro_bias = np.array([float(calibration['gyro_bias_x_dps']),
                          float(calibration['gyro_bias_y_dps']),
                          float(calibration['gyro_bias_z_dps'])])
    gyro_corr = gyro - gyro_bias
    gyro_mag = np.linalg.norm(gyro_corr, axis=1)
    out['gyro_mag_median_dps'] = float(np.median(gyro_mag))
    out['gyro_mag_p95_dps'] = float(np.percentile(gyro_mag, 95, method='linear'))

    if not np.isfinite(list(out.values())).all():
        raise ValueError('Feature extraction produced a non-finite value.')
    return out

def create_feature_tables(samples_df, cal_df):
    cal_lookup = cal_df.set_index(CAL_KEYS, drop=False)
    pooled_rows = []
    for key, trial in samples_df.groupby(TRIAL_KEYS, sort=True):
        if len(trial) != WIN or (trial[CAL_KEYS].nunique() != 1).any():
            raise ValueError(f'Inconsistent trial boundary or calibration: {key}.')
        first = trial.iloc[0]
        cal_key = tuple(first[k] for k in CAL_KEYS)
        if cal_key not in cal_lookup.index:
            raise ValueError(f'Missing session calibration for trial {key}: {cal_key}.')
        calibration = cal_lookup.loc[cal_key]
        if isinstance(calibration, pd.DataFrame):
            raise ValueError(f'Ambiguous session calibration for {cal_key}.')
        row = {name: first[name] for name in META_COLS}
        row['trial_id'] = int(row['trial_id'])
        row.update(extract_feature_pool(trial, calibration))
        pooled_rows.append(row)

    pool = pd.DataFrame(pooled_rows).sort_values(TRIAL_KEYS).reset_index(drop=True)
    tables, cols_by_mode = {}, {}
    for mode in FEATURE_SET_ORDER:
        cols = list(FEATURE_SETS[mode])
        table = pool[META_COLS + cols].copy()
        if not np.isfinite(table[cols].to_numpy(float)).all():
            raise ValueError(f'{mode}: non-finite feature value.')
        if not table[TRIAL_KEYS].equals(trial_meta[TRIAL_KEYS]):
            raise ValueError(f'{mode}: feature rows do not match the accepted trial index.')
        tables[mode] = table
        cols_by_mode[mode] = cols
        print(f'{mode}: {len(table)} examples x {len(cols)} features; all finite.')
    return tables, cols_by_mode, pool

features_by_mode, FEATURE_COLS_BY_MODE, feature_pool = create_feature_tables(samples, calibration_used)
FEATURE_COLS = list(dict.fromkeys(
    col for mode in FEATURE_SET_ORDER for col in FEATURE_COLS_BY_MODE[mode]
))
K_VALUES = sorted({len(cols) for cols in FEATURE_COLS_BY_MODE.values()})
print('Primary feature counts present:', K_VALUES)


## 6. Models and leakage-safe selection

The original Logistic Regression family is left in place. The five requested model
families use the declared grids:

- **KNN:** k = 1,3,5,7,9,11,15; uniform/distance weighting; Manhattan/Euclidean.
- **SVM:** linear and RBF kernels with the declared C/gamma values.
- **Random Forest:** 200/500 trees, depth None/5/10, leaf 1/2/4, sqrt/None features.
- **MLP:** the declared small architectures and alpha values with Adam.
- **LDA:** ordinary SVD plus LSQR with the declared shrinkage settings.

Scaling remains inside the pipeline exactly as in the original notebook structure.

The primary selection score is **mean macro-F1 across inner held-out participants**.
Exact ties prefer fewer predictors and then the predeclared configuration order.

Warnings are recorded by fold/configuration. A fitting error stops the run rather
than being converted into a score.


In [ ]:
# ======================================
# 6. Models and helper functions
# ======================================
model_specs = {
    "Logistic Regression": (
        LogisticRegression(
            max_iter=3000,
            random_state=SEED,
        ),
        {
            "model__C": [
                0.1,
                1.0,
                10.0,
            ]
        },
    ),

    "KNN": (
        KNeighborsClassifier(
            metric="minkowski",
        ),
        {
            "model__n_neighbors": [
                1, 3, 5, 7, 9, 11, 15,
            ],
            "model__weights": [
                "uniform", "distance",
            ],
            "model__p": [
                1, 2,
            ],
        },
    ),

    "SVM": (
        SVC(
            probability=False,
            class_weight=None,
        ),
        [
            {
                "model__kernel": ["linear"],
                "model__C": [0.1, 1.0, 10.0, 100.0],
            },
            {
                "model__kernel": ["rbf"],
                "model__C": [0.1, 1.0, 10.0, 100.0],
                "model__gamma": ["scale", 0.01, 0.1, 1.0],
            },
        ],
    ),

    "Random Forest": (
        RandomForestClassifier(
            bootstrap=True,
            class_weight=None,
            random_state=SEED,
            n_jobs=-1,
        ),
        {
            "model__n_estimators": [200, 500],
            "model__max_depth": [None, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", None],
        },
    ),

    "LDA": (
        LinearDiscriminantAnalysis(),
        [
            {
                "model__solver": ["svd"],
                "model__shrinkage": [None],
            },
            {
                "model__solver": ["lsqr"],
                "model__shrinkage": [None, "auto", 0.1, 0.5, 0.9],
            },
        ],
    ),

    "Small MLP": (
        MLPClassifier(
            activation="relu",
            solver="adam",
            learning_rate="constant",
            learning_rate_init=0.001,
            max_iter=2000,
            tol=1e-4,
            early_stopping=False,
            random_state=SEED,
        ),
        {
            "model__hidden_layer_sizes": [
                (8,), (16,), (32,), (16, 8),
            ],
            "model__alpha": [
                0.0001, 0.001, 0.01,
            ],
        },
    ),
}

if RUN_MODE == 'smoke':
    model_specs = {
        name: (est, {key: [value] for key, value in next(iter(ParameterGrid(grid))).items()})
        for name, (est, grid) in model_specs.items()
    }

MODEL_ORDER = list(model_specs)
MODE_ORDER = list(FEATURE_SET_ORDER)
model_priority = {name: i for i, name in enumerate(MODEL_ORDER)}
mode_priority = {name: i for i, name in enumerate(MODE_ORDER)}
FIT_COUNTS = {'preprocessing': 0, 'inner_model': 0, 'outer_model': 0, 'final_model': 0}

def make_pipeline(estimator, k=None):
    # SelectKBest was intentionally removed: each F1-F6 set is already a fixed,
    # predeclared predictor block. Scaling behavior is otherwise left unchanged.
    return Pipeline([('scale', StandardScaler()),
                     ('model', clone(estimator))])

def fit_logged(estimator, X, y, context, warning_rows):
    caught = []
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            with threadpool_limits(limits=THREAD_LIMIT):
                estimator.fit(X, y)
    except Exception as exc:
        warning_rows.append({**context, 'category': type(exc).__name__, 'message': str(exc),
                             'fit_failed': True})
        pd.DataFrame(warning_rows).to_csv(OUT / 'last_failed_fit_warnings.csv', index=False)
        raise RuntimeError(f'Fit failed at {context}: {exc}') from exc
    finally:
        for warning in caught:
            warning_rows.append({**context, 'category': warning.category.__name__,
                                 'message': str(warning.message), 'fit_failed': False})
    FIT_COUNTS[context['fit_kind']] += 1
    return estimator

def metric_row(y_true, y_pred):
    return {'accuracy': float(accuracy_score(y_true, y_pred)),
            'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
            'macro_f1': float(f1_score(y_true, y_pred, labels=LABELS,
                                      average='macro', zero_division=0))}

def selection_key(row):
    # Fix 1: macro-F1 is the primary inner-CV selection metric.
    # Exact ties prefer fewer predictors, then the predeclared ordering.
    return (row['inner_macro_f1_mean'], -int(row['k']),
            -mode_priority[row['mode']], -model_priority[row['model']],
            -int(row['param_index']))

def select_best(rows):
    if not rows:
        raise ValueError('No model-selection candidates.')
    return max(rows, key=selection_key)


## 7. Checkpoints, manifests and search functions

Each outer development set uses participant-wise **inner LOSO**. With nine development
participants, there are nine inner folds; each participant is validation exactly once.
The final all-participant search uses ten inner participant-held-out folds.

Checkpoints remain tied to the dataset, feature data, code, settings and software
versions. Changing any of the eight corrected modeling rules creates a new run
fingerprint rather than reusing stale checkpoints.


In [28]:
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Not JSON serializable: {type(value).__name__}')

def atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, sort_keys=True, default=json_default), encoding='utf-8')
    temp.replace(path)

def atomic_joblib(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    joblib.dump(obj, temp, compress=3)
    temp.replace(path)

def slug(text):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', str(text)).strip('_')

def load_checkpoint(path):
    if not RESUME or not Path(path).exists():
        return None
    payload = joblib.load(path)
    if payload.get('run_id') != RUN_ID:
        raise ValueError(f'Checkpoint fingerprint mismatch: {path}')
    return payload

def make_inner_splits(dev, scope):
    # Fix 7: participant-wise inner LOSO. For an outer fold this means 9 folds;
    # for the final all-participant search this means 10 folds.
    n_groups = int(dev['participant'].nunique())
    if n_groups < 2:
        raise ValueError('At least two development participants are required.')
    cv = GroupKFold(n_splits=n_groups)
    splits, manifest = [], []
    seen_validation = []
    dummy_x = np.zeros((len(dev), 1))
    for split_no, (tr, va) in enumerate(cv.split(dummy_x, dev['gesture'], dev['participant']), 1):
        training_pids = sorted(dev.iloc[tr]['participant'].unique())
        validation_pids = sorted(dev.iloc[va]['participant'].unique())
        if len(validation_pids) != 1:
            raise AssertionError('Inner LOSO must hold out exactly one participant.')
        if set(training_pids) & set(validation_pids):
            raise AssertionError('Participant overlap in inner CV.')
        if set(dev.iloc[tr]['gesture']) != set(LABELS) or set(dev.iloc[va]['gesture']) != set(LABELS):
            raise ValueError('An inner fold does not contain every gesture.')
        seen_validation.extend(validation_pids)
        splits.append((tr, va))
        manifest.append({'scope': scope, 'inner_fold': split_no,
                         'train_participants': training_pids, 'validation_participants': validation_pids,
                         'train_examples': len(tr), 'validation_examples': len(va)})
    if sorted(seen_validation) != sorted(dev['participant'].unique()):
        raise AssertionError('Every development participant must be validation exactly once in inner LOSO.')
    atomic_json(manifest, OUT / 'split_manifests' / f'{slug(scope)}.json')
    return splits

def prepare_inner_folds(dev, splits, mode, scope, warning_rows):
    prepared = []
    feature_cols = FEATURE_COLS_BY_MODE[mode]
    for split_no, (tr, va) in enumerate(splits, 1):
        X_train = dev.iloc[tr][feature_cols]
        X_val = dev.iloc[va][feature_cols]
        y_train = dev.iloc[tr]['gesture'].to_numpy()
        y_val = dev.iloc[va]['gesture'].to_numpy()
        context = {'scope': scope, 'mode': mode, 'inner_fold': split_no,
                   'fit_kind': 'preprocessing', 'model': 'StandardScaler',
                   'k': len(feature_cols), 'params_json': '{}'}
        scaler = fit_logged(StandardScaler(), X_train, y_train, context, warning_rows)
        train_scaled, val_scaled = scaler.transform(X_train), scaler.transform(X_val)
        prepared.append({'inner_fold': split_no,
                         'X_train': np.ascontiguousarray(train_scaled),
                         'X_val': np.ascontiguousarray(val_scaled),
                         'y_train': y_train, 'y_val': y_val,
                         'val_participants': dev.iloc[va]['participant'].to_numpy()})
    return prepared

def grouped_cv_score(prepared, estimator, k, params, context, warning_rows):
    person_scores, fold_accuracies = [], []
    start_warning = len(warning_rows)
    for fold in prepared:
        model = clone(estimator).set_params(
            **{key.removeprefix('model__'): value for key, value in params.items()}
        )
        fit_logged(model, fold['X_train'], fold['y_train'],
                   {**context, 'inner_fold': fold['inner_fold'], 'fit_kind': 'inner_model'}, warning_rows)
        with threadpool_limits(limits=THREAD_LIMIT):
            pred = model.predict(fold['X_val'])
        fold_accuracies.append(float(accuracy_score(fold['y_val'], pred)))
        for pid in np.unique(fold['val_participants']):
            mask = fold['val_participants'] == pid
            person_scores.append(metric_row(fold['y_val'][mask], pred[mask]))
    accuracy_values = [row['accuracy'] for row in person_scores]
    return {'inner_accuracy_mean': float(np.mean(accuracy_values)),
            'inner_accuracy_sd': float(np.std(accuracy_values, ddof=1)),
            'inner_macro_f1_mean': float(np.mean([row['macro_f1'] for row in person_scores])),
            'inner_balanced_accuracy_mean': float(np.mean([row['balanced_accuracy'] for row in person_scores])),
            'inner_fold_accuracy_sd': float(np.std(fold_accuracies, ddof=1)),
            'fit_warning_count': len(warning_rows) - start_warning}

def search_development(development_pids, scope):
    all_rows, best_k_rows, warning_rows = [], [], []
    base = features_by_mode[MODE_ORDER[0]]
    base_dev = base.loc[base['participant'].isin(development_pids)].reset_index(drop=True)
    if set(base_dev['participant']) != set(development_pids):
        raise ValueError('Development participant list does not match feature data.')
    splits = make_inner_splits(base_dev, scope)

    for mode in MODE_ORDER:
        dev = features_by_mode[mode].loc[
            features_by_mode[mode]['participant'].isin(development_pids)
        ].reset_index(drop=True)
        if not dev[TRIAL_KEYS].equals(base_dev[TRIAL_KEYS]):
            raise AssertionError('Feature-set trial order mismatch.')

        feature_cols = FEATURE_COLS_BY_MODE[mode]
        k = len(feature_cols)
        prepared = None
        prep_path = CHECKPOINT_DIR / slug(scope) / mode / 'preprocessing_warnings.joblib'
        prep_payload = load_checkpoint(prep_path)
        preprocessing_warnings = prep_payload['warnings'] if prep_payload else []

        for model_name in MODEL_ORDER:
            path = CHECKPOINT_DIR / slug(scope) / mode / f'{slug(model_name)}.joblib'
            cached = load_checkpoint(path)
            if cached is not None:
                all_rows.extend(cached['all_rows'])
                best_k_rows.extend(cached['best_k_rows'])
                warning_rows.extend(cached['warnings'])
                print(f'  {scope} | {mode} | {model_name}: checkpoint loaded', flush=True)
                continue

            if prepared is None:
                preprocessing_warnings = []
                prepared = prepare_inner_folds(dev, splits, mode, scope, preprocessing_warnings)
                atomic_joblib({'run_id': RUN_ID, 'warnings': preprocessing_warnings}, prep_path)

            estimator, param_grid = model_specs[model_name]
            unit_rows, unit_warnings = [], []
            started = time.perf_counter()
            for param_index, params in enumerate(ParameterGrid(param_grid)):
                context = {'scope': scope, 'mode': mode, 'model': model_name,
                           'k': k, 'params_json': json.dumps(params, sort_keys=True)}
                metrics = grouped_cv_score(prepared, estimator, k, params, context, unit_warnings)
                row = {'scope': scope, 'mode': mode, 'model': model_name, 'k': k,
                       'params': params, 'param_index': param_index, **metrics}
                unit_rows.append(row)

            unit_best = [select_best(unit_rows)]
            print(f'  {scope} | {mode} | {model_name}: '
                  f'{len(unit_rows)} configurations; '
                  f'{time.perf_counter()-started:.1f}s; {len(unit_warnings)} warnings', flush=True)
            atomic_joblib({'run_id': RUN_ID, 'all_rows': unit_rows, 'best_k_rows': unit_best,
                           'warnings': unit_warnings}, path)
            all_rows.extend(unit_rows)
            best_k_rows.extend(unit_best)
            warning_rows.extend(unit_warnings)

        warning_rows.extend(preprocessing_warnings)

    return {'all_rows': all_rows, 'best_k_rows': best_k_rows, 'warnings': warning_rows}

def evaluate_candidate(candidate, test_pid, warning_rows):
    features = features_by_mode[candidate['mode']]
    feature_cols = FEATURE_COLS_BY_MODE[candidate['mode']]
    dev = features.loc[features['participant'].ne(test_pid)]
    test = features.loc[features['participant'].eq(test_pid)]
    if set(dev['participant']) & set(test['participant']):
        raise AssertionError('Outer participant overlap.')

    pipe = make_pipeline(model_specs[candidate['model']][0], candidate['k'])
    pipe.set_params(**candidate['params'])
    context = {'scope': f'outer_{test_pid}', 'mode': candidate['mode'], 'model': candidate['model'],
               'k': candidate['k'], 'fit_kind': 'outer_model', 'inner_fold': None,
               'params_json': json.dumps(candidate['params'], sort_keys=True)}
    start_warning = len(warning_rows)
    fit_logged(pipe, dev[feature_cols], dev['gesture'], context, warning_rows)
    with threadpool_limits(limits=THREAD_LIMIT):
        pred = pipe.predict(test[feature_cols])
    metric = metric_row(test['gesture'], pred)

    result = {'test_participant': test_pid, 'selected_mode': candidate['mode'],
              'selected_model': candidate['model'], 'selected_k': int(candidate['k']),
              'selected_params': candidate['params'], 'selected_features': list(feature_cols),
              'inner_cv_accuracy': candidate['inner_accuracy_mean'],
              'inner_cv_macro_f1': candidate['inner_macro_f1_mean'],
              'outer_test_accuracy': metric['accuracy'],
              'outer_test_balanced_accuracy': metric['balanced_accuracy'],
              'outer_test_macro_f1': metric['macro_f1'],
              'outer_fit_warning_count': len(warning_rows) - start_warning}
    predictions = test[TRIAL_KEYS + ['session', 'ref_id']].copy()
    predictions['prediction'] = pred
    predictions['correct'] = predictions['gesture'].eq(predictions['prediction'])
    predictions['selected_mode'] = candidate['mode']
    predictions['selected_model'] = candidate['model']
    predictions['selected_k'] = candidate['k']
    return result, predictions

def code_fingerprint(code_object):
    # Exclude cell execution counters, filenames and line numbers, so rerunning
    # an unchanged Colab notebook can reuse checkpoints.
    constants = [code_fingerprint(c) if isinstance(c, types.CodeType) else repr(c)
                 for c in code_object.co_consts]
    return {'bytecode': code_object.co_code.hex(), 'constants': constants,
            'names': code_object.co_names, 'variables': code_object.co_varnames,
            'freevars': code_object.co_freevars, 'cellvars': code_object.co_cellvars,
            'args': code_object.co_argcount, 'kwonly': code_object.co_kwonlyargcount,
            'flags': code_object.co_flags}

CORE_FUNCTIONS = [load_accepted_trials, build_calibration_table, extract_feature_pool,
                  create_feature_tables, make_pipeline, fit_logged, metric_row,
                  selection_key, select_best, make_inner_splits, prepare_inner_folds,
                  grouped_cv_score, search_development, evaluate_candidate]
function_digest = hashlib.sha256(json.dumps(
    {f.__name__: code_fingerprint(f.__code__) for f in CORE_FUNCTIONS},
    sort_keys=True).encode()).hexdigest()

feature_digest = hashlib.sha256()
for mode in MODE_ORDER:
    cols = FEATURE_COLS_BY_MODE[mode]
    frame = features_by_mode[mode][TRIAL_KEYS + cols]
    feature_digest.update(pd.util.hash_pandas_object(frame, index=False).values.tobytes())

settings = {'pipeline_version': PIPELINE_VERSION, 'run_mode': RUN_MODE,
            'dataset_sha256': DATASET_SHA256, 'feature_data_sha256': feature_digest.hexdigest(),
            'function_sha256': function_digest, 'seed': SEED,
            'inner_cv': 'participant_LOSO',
            'trial_samples': WIN, 'sample_period_ms': SAMPLE_PERIOD_MS,
            'acc_lsb_per_g': ACC_LSB_PER_G, 'gyro_lsb_per_dps': GYRO_LSB_PER_DPS,
            'span_zero_epsilon': SPAN_ZERO_EPS,
            'span_policy': 'diagnostic only; excluded from primary F1-F6 feature sets',
            'feature_set_order': MODE_ORDER,
            'feature_sets': FEATURE_COLS_BY_MODE,
            'model_order': MODEL_ORDER,
            'models': {name: {'estimator_params': est.get_params(), 'grid': grid}
                       for name, (est, grid) in model_specs.items()},
            'selection_metric': 'mean inner-held-out participant macro F1',
            'versions': VERSIONS, 'thread_limit': THREAD_LIMIT}
RUN_ID = hashlib.sha256(json.dumps(settings, sort_keys=True, default=json_default).encode()).hexdigest()
OUT = OUTPUT_BASE / f'{RUN_MODE}_{RUN_ID[:16]}'
OUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

atomic_json({**settings, 'run_id': RUN_ID}, OUT / 'settings.json')
for mode, table in features_by_mode.items():
    table.to_csv(OUT / f'trial_features_{mode}.csv', index=False)
trial_meta.to_csv(OUT / 'accepted_trial_index.csv', index=False)
calibration_diagnostics.to_csv(OUT / 'calibration_reference_diagnostics.csv', index=False)
calibration_used.to_csv(OUT / 'calibration_references_used.csv', index=False)

grid_count = sum(len(ParameterGrid(grid)) for _, grid in model_specs.values())
outer_inner_fits = len(participants) * len(MODE_ORDER) * grid_count * (len(participants) - 1)
outer_refit_count = len(participants) * len(MODE_ORDER) * len(MODEL_ORDER)
final_inner_fits = len(MODE_ORDER) * grid_count * len(participants)
print('Run fingerprint:', RUN_ID)
print('Results and checkpoints:', OUT.resolve())
print('Outer evaluation classifier fits:', outer_inner_fits + outer_refit_count)
print('Additional final tuning/refit fits:', final_inner_fits + 1 if TRAIN_FINAL_MODEL else 0)


NameError: name 'extract_feature_pool' is not defined

## 8. Run nested participant-held-out evaluation

For each outer participant, all choices are made using development participants only:

1. **Overall selection procedure:** tune feature set, model family and model parameters.
2. **Each classifier:** keep model family fixed; tune feature set and parameters.
3. **Each feature-set/classifier pair:** keep both fixed; tune model parameters.

Outer-test performance is never used by `select_best`.


In [ ]:
experiment_start = time.perf_counter()
outer_payloads = []
for fold_number, test_pid in enumerate(participants, 1):
    print(f'\nOUTER FOLD {fold_number}/{len(participants)} — held out: {test_pid}', flush=True)
    outer_path = CHECKPOINT_DIR / f'complete_outer_{test_pid}.joblib'
    payload = load_checkpoint(outer_path)
    if payload is None:
        development_pids = [pid for pid in participants if pid != test_pid]
        assert test_pid not in development_pids
        search = search_development(development_pids, f'outer_{test_pid}')
        candidates = search['best_k_rows']
        # Lock all selections before calculating any outer-test performance.
        overall_choice = select_best(candidates)
        model_choices = {model: select_best([r for r in candidates if r['model'] == model])
                         for model in MODEL_ORDER}
        pair_choices = {(mode, model): select_best([r for r in candidates
                                                    if r['mode'] == mode and r['model'] == model])
                        for mode in MODE_ORDER for model in MODEL_ORDER}
        outer_warnings, evaluated = [], {}
        for pair, choice in pair_choices.items():
            evaluated[pair] = evaluate_candidate(choice, test_pid, outer_warnings)
        overall_result, overall_predictions = evaluated[(overall_choice['mode'], overall_choice['model'])]
        model_results, model_predictions = [], []
        for model, choice in model_choices.items():
            result, prediction = evaluated[(choice['mode'], model)]
            model_results.append(result)
            model_predictions.append(prediction)
        payload = {'run_id': RUN_ID, 'test_participant': test_pid,
                   'outer_result': overall_result, 'outer_predictions': overall_predictions,
                   'model_results': model_results, 'model_predictions': pd.concat(model_predictions, ignore_index=True),
                   'pair_results': [item[0] for item in evaluated.values()],
                   'pair_predictions': pd.concat([item[1] for item in evaluated.values()], ignore_index=True),
                   'inner_all': search['all_rows'], 'inner_best_k': candidates,
                   'warnings': search['warnings'] + outer_warnings}
        atomic_joblib(payload, outer_path)
    else:
        print('  Complete outer-fold checkpoint loaded; no refitting.', flush=True)
    outer_payloads.append(payload)
    result = payload['outer_result']
    print(f"  Inner selection: {result['selected_model']} | {result['selected_mode']} | "
          f"features={result['selected_k']} | outer accuracy={100*result['outer_test_accuracy']:.2f}%", flush=True)

outer_results = pd.DataFrame([p['outer_result'] for p in outer_payloads])
model_outer_results = pd.DataFrame([r for p in outer_payloads for r in p['model_results']])
pair_outer_results = pd.DataFrame([r for p in outer_payloads for r in p['pair_results']])
inner_results = pd.DataFrame([r for p in outer_payloads for r in p['inner_best_k']])
inner_all_results = pd.DataFrame([r for p in outer_payloads for r in p['inner_all']])
outer_predictions = pd.concat([p['outer_predictions'] for p in outer_payloads], ignore_index=True)
model_outer_predictions = pd.concat([p['model_predictions'] for p in outer_payloads], ignore_index=True)
pair_outer_predictions = pd.concat([p['pair_predictions'] for p in outer_payloads], ignore_index=True)
WARNING_COLUMNS = ['scope', 'mode', 'model', 'k', 'params_json', 'inner_fold', 'fit_kind',
                   'category', 'message', 'fit_failed']
warning_rows = [r for p in outer_payloads for r in p['warnings']]
warning_log = pd.DataFrame(warning_rows, columns=WARNING_COLUMNS)
warning_log.to_csv(OUT / 'fit_warnings.csv', index=False)

assert len(outer_predictions) == len(trial_meta)
assert not outer_predictions[TRIAL_KEYS].duplicated().any()
assert outer_predictions.sort_values(TRIAL_KEYS)[TRIAL_KEYS].reset_index(drop=True).equals(trial_meta[TRIAL_KEYS])
assert len(model_outer_predictions) == len(trial_meta) * len(MODEL_ORDER)
assert len(pair_outer_predictions) == len(trial_meta) * len(MODEL_ORDER) * len(MODE_ORDER)
assert not model_outer_predictions[TRIAL_KEYS + ['selected_model']].duplicated().any()
assert not pair_outer_predictions[TRIAL_KEYS + ['selected_model', 'selected_mode']].duplicated().any()
print(f'\nNested evaluation finished in {time.perf_counter()-experiment_start:.1f} seconds this execution.')
print('Fit counters this execution (resumed fits are not recounted):', FIT_COUNTS)
print('Recorded warning events:', len(warning_log))
if not warning_log.empty:
    display(warning_log.groupby(['category', 'model']).size().rename('events').reset_index())


## 9. Outer-test performance

Report the mean participant-held-out accuracy and SD across participants for the
overall selection procedure, together with macro-F1. Classifier and fixed
feature-set/classifier comparisons use the same held-out participants.

The **selection** criterion inside development data is macro-F1; accuracy is still
reported as an outcome metric.


In [ ]:
display(outer_results[['test_participant', 'selected_mode', 'selected_model', 'selected_k',
                       'outer_test_accuracy', 'outer_test_macro_f1']])
mean_accuracy = float(outer_results['outer_test_accuracy'].mean())
sd_accuracy = float(outer_results['outer_test_accuracy'].std(ddof=1))
mean_macro_f1 = float(outer_results['outer_test_macro_f1'].mean())
sd_macro_f1 = float(outer_results['outer_test_macro_f1'].std(ddof=1))
pooled = metric_row(outer_predictions['gesture'], outer_predictions['prediction'])
primary_summary = pd.DataFrame([{'procedure': 'Overall inner-CV selection',
                                 'run_mode': RUN_MODE, 'participants': len(participants),
                                 'trials': len(trial_meta), 'mean_outer_accuracy': mean_accuracy,
                                 'sd_outer_accuracy': sd_accuracy, 'mean_outer_macro_f1': mean_macro_f1,
                                 'sd_outer_macro_f1': sd_macro_f1,
                                 'pooled_outer_accuracy': pooled['accuracy'],
                                 'pooled_outer_macro_f1': pooled['macro_f1']}])
print(f'[{RUN_MODE.upper()}] Overall mean outer accuracy: {100*mean_accuracy:.2f}% '
      f'± {100*sd_accuracy:.2f} percentage points (participant SD)')
print(f'Mean participant macro-F1: {mean_macro_f1:.4f}; pooled macro-F1: {pooled["macro_f1"]:.4f}')

def summarize_outer(frame, grouping):
    return (frame.groupby(grouping, as_index=False).agg(
        participants=('test_participant', 'nunique'),
        mean_outer_accuracy=('outer_test_accuracy', 'mean'),
        sd_outer_accuracy=('outer_test_accuracy', 'std'),
        mean_outer_macro_f1=('outer_test_macro_f1', 'mean'),
        sd_outer_macro_f1=('outer_test_macro_f1', 'std'),
        mean_selected_feature_count=('selected_k', 'mean')))

model_summary = summarize_outer(model_outer_results, ['selected_model'])
pair_summary = summarize_outer(pair_outer_results, ['selected_mode', 'selected_model'])
print('Classifier comparison: feature set/parameters selected within inner CV.')
display(model_summary)
print('Fixed feature-set/classifier comparison: parameters selected within inner CV.')
display(pair_summary)

fig, ax = plt.subplots(figsize=(10, 5))
for i, model in enumerate(MODEL_ORDER):
    d = model_outer_results.loc[model_outer_results['selected_model'].eq(model)]
    ax.scatter(np.full(len(d), i), 100*d['outer_test_accuracy'], alpha=.6, s=30)
    ax.errorbar(i, 100*d['outer_test_accuracy'].mean(),
                yerr=100*d['outer_test_accuracy'].std(ddof=1), fmt='ks', capsize=5)
ax.set_xticks(range(len(MODEL_ORDER)), MODEL_ORDER, rotation=20, ha='right')
ax.set_ylabel('Held-out participant accuracy (%)')
ax.set_title(f'{RUN_MODE.upper()}: each classifier — participants, mean and SD')
ax.grid(axis='y', alpha=.25)
fig.tight_layout()
fig.savefig(OUT / 'outer_classifier_comparison.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)

matrix = pair_summary.pivot(index='selected_model', columns='selected_mode', values='mean_outer_accuracy')
matrix = matrix.reindex(index=MODEL_ORDER, columns=MODE_ORDER)
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(100*matrix.to_numpy(), vmin=0, vmax=100, cmap='Blues')
ax.set_xticks(range(len(MODE_ORDER)), MODE_ORDER)
ax.set_yticks(range(len(MODEL_ORDER)), MODEL_ORDER)
for i in range(len(MODEL_ORDER)):
    for j in range(len(MODE_ORDER)):
        value = 100*matrix.iloc[i, j]
        ax.text(j, i, f'{value:.1f}%', ha='center', va='center', color='white' if value > 60 else 'black')
ax.set_title(f'{RUN_MODE.upper()}: mean outer accuracy; feature set/model parameters tuned inside each fold')
fig.colorbar(im, ax=ax, label='Accuracy (%)')
fig.tight_layout()
fig.savefig(OUT / 'outer_representation_classifier_comparison.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)


## 10. Inner-CV performance across F1-F6

The following summaries describe development-set inner-CV performance for the six
fixed primary feature sets. They are not independent outer-test results and they do
not overwrite the nested selections already made.


In [ ]:
# Inner-CV descriptive performance by the six fixed primary feature sets.
# These summaries are not outer-test accuracy and do not change the selections already made.
feature_set_summary = inner_results.groupby(['mode', 'model'], as_index=False).agg(
    mean_inner_accuracy=('inner_accuracy_mean', 'mean'),
    sd_across_outer_development_sets=('inner_accuracy_mean', 'std'),
    mean_inner_macro_f1=('inner_macro_f1_mean', 'mean'),
    feature_count=('k', 'first'))

best_feature_rows = []
for model_name in MODEL_ORDER:
    d = feature_set_summary.loc[feature_set_summary['model'].eq(model_name)].copy()
    d['mode_order'] = d['mode'].map(mode_priority)
    best = d.sort_values(['mean_inner_macro_f1', 'feature_count', 'mode_order'],
                         ascending=[False, True, True]).iloc[0]
    best_feature_rows.append({
        'model': model_name,
        'descriptive_best_feature_set': best['mode'],
        'feature_count': int(best['feature_count']),
        'mean_inner_macro_f1': float(best['mean_inner_macro_f1']),
        'mean_inner_accuracy': float(best['mean_inner_accuracy']),
    })

best_feature_summary = pd.DataFrame(best_feature_rows)
display(feature_set_summary)
display(best_feature_summary)

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(MODE_ORDER))
for model_name in MODEL_ORDER:
    d = feature_set_summary.loc[feature_set_summary['model'].eq(model_name)].set_index('mode').reindex(MODE_ORDER)
    ax.plot(x, d['mean_inner_macro_f1'], marker='o', label=model_name)
ax.set_xticks(x, MODE_ORDER, rotation=35, ha='right')
ax.set_ylabel('Mean inner-held-out participant macro-F1')
ax.set_title(f'{RUN_MODE.upper()}: inner-CV performance across fixed F1-F6 feature sets')
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'inner_macro_f1_by_feature_set.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)


## 11. Selection stability

Counts show how often the selected F1-F6 feature set, classifier and exact predictor
block change when a different participant is held out. Feature frequency is
descriptive and is not used to create another feature set from outer outcomes.


In [ ]:
for column in ['selected_mode', 'selected_model', 'selected_k']:
    print(column)
    display(outer_results[column].value_counts().sort_index().rename('outer_folds').to_frame())
selected_feature_rows = []
for _, row in outer_results.iterrows():
    for feature in row['selected_features']:
        selected_feature_rows.append({'test_participant': row['test_participant'],
                                      'mode': row['selected_mode'], 'model': row['selected_model'],
                                      'k': row['selected_k'], 'feature': feature})
selected_feature_table = pd.DataFrame(selected_feature_rows)
feature_frequency = selected_feature_table['feature'].value_counts().reindex(FEATURE_COLS, fill_value=0)
feature_frequency = feature_frequency.rename_axis('feature').rename('times_selected_across_outer_folds').to_frame()
display(feature_frequency.sort_values('times_selected_across_outer_folds', ascending=False))
fig, ax = plt.subplots(figsize=(9, 9))
feature_frequency.sort_values('times_selected_across_outer_folds').plot.barh(ax=ax, legend=False)
ax.set_xlabel('Number of outer folds selecting the feature')
ax.set_title(f'{RUN_MODE.upper()}: selected feature frequency — overall procedure')
fig.tight_layout()
fig.savefig(OUT / 'selected_feature_frequency.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)


## 12. Pooled outer predictions, confusion matrices and per-class reports

Every pooled prediction was obtained with that participant held outside model
fitting and model selection. Calibration is the separately declared exception:
the current participant's own pre-gesture reference is available when the chosen
representation requires it. Reports are saved for the overall procedure and all
six classifier families.


In [ ]:
def prediction_report(predictions):
    return pd.DataFrame(classification_report(
        predictions['gesture'], predictions['prediction'], labels=LABELS,
        target_names=[f'{g}: {gesture_names[g]}' for g in LABELS],
        output_dict=True, zero_division=0)).T

def draw_confusion(ax, predictions, title):
    cm = confusion_matrix(predictions['gesture'], predictions['prediction'], labels=LABELS)
    ax.imshow(cm, cmap='Blues', vmin=0)
    ax.set_xticks(range(len(LABELS)), LABELS, rotation=45, ha='right')
    ax.set_yticks(range(len(LABELS)), LABELS)
    ax.set_xlabel('Predicted gesture')
    ax.set_ylabel('True gesture')
    ax.set_title(title)
    threshold = cm.max()/2
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8,
                    color='white' if cm[i, j] > threshold else 'black')
    return cm

report_df = prediction_report(outer_predictions)
display(report_df)
fig, ax = plt.subplots(figsize=(8, 7))
cm = draw_confusion(ax, outer_predictions, f'{RUN_MODE.upper()}: overall selection — outer predictions')
fig.tight_layout()
fig.savefig(OUT / 'pooled_outer_confusion_matrix.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(OUT / 'pooled_outer_confusion_matrix.csv')
model_reports = {}
fig, axes = plt.subplots(2, 3, figsize=(19, 12))
for ax, model_name in zip(axes.flat, MODEL_ORDER):
    prediction = model_outer_predictions.loc[model_outer_predictions['selected_model'].eq(model_name)]
    model_reports[model_name] = prediction_report(prediction)
    model_cm = draw_confusion(ax, prediction, model_name)
    pd.DataFrame(model_cm, index=LABELS, columns=LABELS).to_csv(OUT / f'confusion_{slug(model_name)}.csv')
fig.suptitle(f'{RUN_MODE.upper()}: outer-test confusion matrices, separately tuned classifiers')
fig.tight_layout(rect=[0, 0, 1, .97])
fig.savefig(OUT / 'all_classifier_outer_confusion_matrices.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)


## 13. Train and export one final model using all participants

After nested evaluation, run the same declared F1-F6/model search using all ten
participants as development data with participant-wise inner LOSO. Select the
feature set/model/parameters using mean inner macro-F1, then refit once on all
500 accepted trials.

This final refit is a deployment artifact; it does not create a new independent
test score.


In [ ]:
final_training_warnings = []
final_bundle = None
final_search_results = pd.DataFrame()
final_search_all_results = pd.DataFrame()
if TRAIN_FINAL_MODEL:
    final_path = OUT / 'final_model_bundle.joblib'
    saved = load_checkpoint(final_path)
    if saved is not None:
        final_bundle = saved
        final_training_warnings = saved['training_warnings']
        final_search_results = pd.DataFrame(saved['search_best_k'])
        final_search_all_results = pd.DataFrame(saved['search_all'])
        print('Final all-participant model checkpoint loaded.')
    else:
        final_search = search_development(participants, 'all_participants')
        chosen = select_best(final_search['best_k_rows'])
        final_training_warnings = list(final_search['warnings'])
        train = features_by_mode[chosen['mode']]
        feature_cols = FEATURE_COLS_BY_MODE[chosen['mode']]
        final_pipeline = make_pipeline(model_specs[chosen['model']][0], chosen['k'])
        final_pipeline.set_params(**chosen['params'])
        fit_logged(final_pipeline, train[feature_cols], train['gesture'],
                   {'scope': 'all_participants_final_refit', 'mode': chosen['mode'],
                    'model': chosen['model'], 'k': chosen['k'], 'inner_fold': None,
                    'params_json': json.dumps(chosen['params'], sort_keys=True), 'fit_kind': 'final_model'},
                   final_training_warnings)
        metadata = {'run_id': RUN_ID, 'pipeline_version': PIPELINE_VERSION, 'run_mode': RUN_MODE,
                    'selected_mode': chosen['mode'], 'selected_model': chosen['model'],
                    'selected_k': int(chosen['k']), 'selected_params': chosen['params'],
                    'feature_columns': feature_cols,
                    'selected_features': list(feature_cols),
                    'gesture_names': gesture_names, 'training_participants': participants,
                    'training_trials': len(train), 'window_samples': WIN,
                    'sample_period_ms': SAMPLE_PERIOD_MS, 'acc_lsb_per_g': ACC_LSB_PER_G,
                    'gyro_lsb_per_dps': GYRO_LSB_PER_DPS, 'dataset_sha256': DATASET_SHA256,
                    'versions': VERSIONS,
                    'inner_tuning_macro_f1_not_test_performance': chosen['inner_macro_f1_mean'],
                    'calibration_requirement': (
                        'None beyond the normal acquisition protocol.'
                        if chosen['mode'] == 'F1_RAW_FLEX'
                        else 'Current-session neutral calibration recorded before prediction.'
                    )}
        final_bundle = {'run_id': RUN_ID, 'pipeline': final_pipeline, 'metadata': metadata,
                        'training_warnings': final_training_warnings,
                        'search_best_k': final_search['best_k_rows'], 'search_all': final_search['all_rows']}
        atomic_joblib(final_bundle, final_path)
        final_search_results = pd.DataFrame(final_search['best_k_rows'])
        final_search_all_results = pd.DataFrame(final_search['all_rows'])
    atomic_json(final_bundle['metadata'], OUT / 'final_model_metadata.json')
    print('Final configuration:', {k: final_bundle['metadata'][k] for k in
                                    ['selected_mode', 'selected_model', 'selected_k', 'selected_params']})
    print('Saved all-participant pipeline:', final_path)
    print('This refit has no new independent test accuracy.')
else:
    print('Final tuning/refit skipped because TRAIN_FINAL_MODEL=False.')


## 14. Export prediction helper and verify training/inference agreement

The exported helper reproduces the same F1-F6 feature definitions. F1 requires only
the current 50-sample trial. F2-F6 require the current session neutral calibration;
F5 additionally uses the session gyro bias. No span normalization is used in the
primary inference path.

The notebook verifies feature equality between the training code and exported helper
for all six feature sets before checking a reloaded-model prediction.


In [ ]:
INFERENCE_MODULE_SOURCE = r'''"""SmartGlove Python inference for the locked primary F1-F6 feature sets.
Use the package versions in inference_requirements.txt.
The returned prediction is a gesture ID; metadata contains the gesture names.
Calibration inputs must come from the current user's session, before prediction.
"""
from pathlib import Path
import numpy as np
import pandas as pd
import joblib

WIN = 50
SAMPLE_PERIOD_MS = 40
ACC_LSB_PER_G = 8192.0
GYRO_LSB_PER_DPS = 16.4
FLEX = ['flex_thumb', 'flex_index', 'flex_middle', 'flex_ring', 'flex_little']
ACC = ['ax', 'ay', 'az']
GYRO = ['gx', 'gy', 'gz']
QUAT = ['qw', 'qx', 'qy', 'qz']
RAW_SENSOR_COLS = FLEX + ACC + GYRO + QUAT

FEATURE_SETS = {
    'F1_RAW_FLEX': [f'{f}_median' for f in FLEX],
    'F2_NEUTRAL_FLEX': [f'{f}_centered_median' for f in FLEX],
    'F3_RAW_FLEX_GRAVITY': [f'{f}_median' for f in FLEX] + ['gravity_rel_x', 'gravity_rel_y', 'gravity_rel_z'],
    'F4_NEUTRAL_FLEX_GRAVITY': [f'{f}_centered_median' for f in FLEX] + ['gravity_rel_x', 'gravity_rel_y', 'gravity_rel_z'],
    'F5_NEUTRAL_FLEX_STATIC_IMU': [f'{f}_centered_median' for f in FLEX]
        + ['gravity_rel_x', 'gravity_rel_y', 'gravity_rel_z']
        + ['acc_mag_median_g', 'acc_mag_sd_g', 'gyro_mag_median_dps', 'gyro_mag_p95_dps'],
    'F6_NEUTRAL_FLEX_IQR': [f'{f}_centered_median' for f in FLEX] + [f'{f}_iqr' for f in FLEX],
}

def calibration_from_recordings(neutral_samples, gyro_bias_dps=None):
    """Construct a session calibration dictionary.

    For exact parity with training, pass the firmware/session gyro bias in deg/s
    when available. If omitted, the mean exported neutral gyro is used.
    """
    required = FLEX + ACC + GYRO
    if len(neutral_samples) < 2 or not np.isfinite(neutral_samples[required].to_numpy(float)).all():
        raise ValueError('Neutral calibration must contain finite flex/IMU samples.')
    result = {}
    for finger in FLEX:
        result[f'{finger}_neutral'] = float(neutral_samples[finger].mean())

    acc = neutral_samples[ACC].to_numpy(float) / ACC_LSB_PER_G
    acc_mean = acc.mean(axis=0)
    result['neutral_accel_x_g'] = float(acc_mean[0])
    result['neutral_accel_y_g'] = float(acc_mean[1])
    result['neutral_accel_z_g'] = float(acc_mean[2])

    if gyro_bias_dps is None:
        bias = (neutral_samples[GYRO].to_numpy(float) / GYRO_LSB_PER_DPS).mean(axis=0)
    else:
        bias = np.asarray(gyro_bias_dps, dtype=float)
        if bias.shape != (3,):
            raise ValueError('gyro_bias_dps must contain exactly three values.')
    if not np.isfinite(bias).all():
        raise ValueError('Non-finite gyro bias.')
    result['gyro_bias_x_dps'] = float(bias[0])
    result['gyro_bias_y_dps'] = float(bias[1])
    result['gyro_bias_z_dps'] = float(bias[2])
    return result

def extract_feature_pool(trial, calibration=None):
    if len(trial) != WIN:
        raise ValueError(f'Expected {WIN} samples, got {len(trial)}.')
    if not np.isfinite(trial[RAW_SENSOR_COLS].to_numpy(float)).all():
        raise ValueError('Non-finite sensor input to feature extraction.')
    if 'timestamp_ms' in trial.columns:
        timestamps = trial['timestamp_ms'].to_numpy(float)
        if not np.isfinite(timestamps).all() or not np.all(np.diff(timestamps) == SAMPLE_PERIOD_MS):
            raise ValueError('Trial timestamps must have 40 ms spacing.')

    out = {}
    flex = trial[FLEX].to_numpy(float)
    flex_median = np.median(flex, axis=0)
    q25 = np.percentile(flex, 25, axis=0, method='linear')
    q75 = np.percentile(flex, 75, axis=0, method='linear')
    for i, finger in enumerate(FLEX):
        out[f'{finger}_median'] = float(flex_median[i])
        out[f'{finger}_iqr'] = float(q75[i] - q25[i])

    if calibration is None:
        return out

    neutral_flex = np.array([float(calibration[f'{f}_neutral']) for f in FLEX])
    centered = flex_median - neutral_flex
    for i, finger in enumerate(FLEX):
        out[f'{finger}_centered_median'] = float(centered[i])

    acc = trial[ACC].to_numpy(float) / ACC_LSB_PER_G
    a_med = np.median(acc, axis=0)
    a_norm = float(np.linalg.norm(a_med))
    if a_norm <= 0:
        raise ValueError('Trial median acceleration vector has zero norm.')
    u = a_med / a_norm

    u0 = np.array([float(calibration['neutral_accel_x_g']),
                   float(calibration['neutral_accel_y_g']),
                   float(calibration['neutral_accel_z_g'])])
    u0_norm = float(np.linalg.norm(u0))
    if u0_norm <= 0:
        raise ValueError('Neutral acceleration vector has zero norm.')
    u_rel = u - (u0 / u0_norm)
    for name, value in zip(['gravity_rel_x', 'gravity_rel_y', 'gravity_rel_z'], u_rel):
        out[name] = float(value)

    acc_mag = np.linalg.norm(acc, axis=1)
    out['acc_mag_median_g'] = float(np.median(acc_mag))
    out['acc_mag_sd_g'] = float(acc_mag.std(ddof=1))

    gyro = trial[GYRO].to_numpy(float) / GYRO_LSB_PER_DPS
    bias = np.array([float(calibration['gyro_bias_x_dps']),
                     float(calibration['gyro_bias_y_dps']),
                     float(calibration['gyro_bias_z_dps'])])
    gyro_mag = np.linalg.norm(gyro - bias, axis=1)
    out['gyro_mag_median_dps'] = float(np.median(gyro_mag))
    out['gyro_mag_p95_dps'] = float(np.percentile(gyro_mag, 95, method='linear'))

    if not np.isfinite(list(out.values())).all():
        raise ValueError('Feature extraction produced a non-finite value.')
    return out

def extract_features_for_set(trial, mode, calibration=None):
    if mode not in FEATURE_SETS:
        raise ValueError(f'Unknown feature set: {mode}')
    if mode != 'F1_RAW_FLEX' and calibration is None:
        raise ValueError(f'{mode} requires the current-session neutral calibration.')
    pool = extract_feature_pool(trial, calibration)
    cols = FEATURE_SETS[mode]
    missing = [c for c in cols if c not in pool]
    if missing:
        raise ValueError(f'Missing required features for {mode}: {missing}')
    return {c: pool[c] for c in cols}

def predict_trial(samples, bundle, calibration=None):
    """Predict one gesture from one completed 50-sample trial."""
    if isinstance(bundle, (str, Path)):
        bundle = joblib.load(bundle)
    metadata = bundle['metadata']
    expected = {'window_samples': WIN, 'sample_period_ms': SAMPLE_PERIOD_MS,
                'acc_lsb_per_g': ACC_LSB_PER_G, 'gyro_lsb_per_dps': GYRO_LSB_PER_DPS}
    for key, value in expected.items():
        if metadata[key] != value:
            raise ValueError(f'Inference constants do not match model metadata: {key}.')
    mode = metadata['selected_mode']
    features = extract_features_for_set(samples, mode, calibration)
    if list(features) != metadata['feature_columns']:
        raise ValueError('Feature schema differs from the trained model.')
    X = pd.DataFrame([features], columns=metadata['feature_columns'])
    return str(bundle['pipeline'].predict(X)[0])
'''

# Update constants from the actual run rather than assuming the defaults were unchanged.
constant_overrides = {'WIN': WIN, 'SAMPLE_PERIOD_MS': SAMPLE_PERIOD_MS,
                      'ACC_LSB_PER_G': ACC_LSB_PER_G, 'GYRO_LSB_PER_DPS': GYRO_LSB_PER_DPS,
                      'FLEX': FLEX, 'ACC': ACC, 'GYRO': GYRO, 'QUAT': QUAT,
                      'RAW_SENSOR_COLS': RAW_SENSOR_COLS, 'FEATURE_SETS': FEATURE_SETS}
inference_source = INFERENCE_MODULE_SOURCE + '\n' + '\n'.join(
    f'{name} = {value!r}' for name, value in constant_overrides.items()) + '\n'
(OUT / 'smartglove_inference.py').write_text(inference_source, encoding='utf-8')
requirements = [f'numpy=={np.__version__}', f'pandas=={pd.__version__}',
                f'scipy=={scipy.__version__}', f'scikit-learn=={sklearn.__version__}',
                f'joblib=={joblib.__version__}']
(OUT / 'inference_requirements.txt').write_text('\n'.join(requirements) + '\n')
inference_namespace = {}
exec(compile(inference_source, 'smartglove_inference.py', 'exec'), inference_namespace)

if final_bundle is not None:
    row = trial_meta.iloc[0]
    raw_trial = samples.loc[samples['participant'].eq(row['participant'])
                            & samples['gesture'].eq(row['gesture'])
                            & samples['trial_id'].eq(row['trial_id'])].copy()
    cal_row = calibration_used.loc[calibration_used['participant'].eq(row['participant'])
                                   & calibration_used['session'].eq(row['session'])
                                   & calibration_used['ref_id'].eq(row['ref_id'])].iloc[0].to_dict()

    # Check all six primary feature sets.
    for mode in MODE_ORDER:
        exported_features = inference_namespace['extract_features_for_set'](
            raw_trial, mode, None if mode == 'F1_RAW_FLEX' else cal_row)
        table = features_by_mode[mode]
        cols = FEATURE_COLS_BY_MODE[mode]
        expected = table.loc[table['participant'].eq(row['participant'])
                              & table['gesture'].eq(row['gesture'])
                              & table['trial_id'].eq(row['trial_id']), cols].iloc[0].to_numpy(float)
        np.testing.assert_allclose(np.asarray(list(exported_features.values())), expected,
                                   rtol=1e-10, atol=1e-10)

    reloaded = joblib.load(OUT / 'final_model_bundle.joblib')
    selected_mode = reloaded['metadata']['selected_mode']
    selected_cal = None if selected_mode == 'F1_RAW_FLEX' else cal_row
    predicted = inference_namespace['predict_trial'](raw_trial, reloaded, selected_cal)
    cols = FEATURE_COLS_BY_MODE[selected_mode]
    expected_features = features_by_mode[selected_mode].loc[
        features_by_mode[selected_mode]['participant'].eq(row['participant'])
        & features_by_mode[selected_mode]['gesture'].eq(row['gesture'])
        & features_by_mode[selected_mode]['trial_id'].eq(row['trial_id']), cols]
    assert predicted == str(reloaded['pipeline'].predict(expected_features)[0])
    print('PASS: exported features match all six notebook feature sets.')
    print('PASS: reloaded model and standalone inference predictions agree.')


## 15. Save all results and download

The ZIP includes result tables, labeled plots, warning logs, split manifests,
checkpoints, settings and (when enabled) the final model/inference helper.
All exports carry the run identity through their containing folder and metadata.

`fit_warnings.csv` contains nested-evaluation warnings;
`final_training_warnings.csv` contains final-search/refit warnings. Empty logs
retain column headers. No candidate with a fitting exception is silently skipped.


In [ ]:
def export_table(frame, filename, index=False):
    out = frame.copy()
    for column in out.columns:
        if out[column].map(lambda v: isinstance(v, (dict, list, tuple))).any():
            out[column] = out[column].map(lambda v: json.dumps(v, sort_keys=True, default=json_default)
                                         if isinstance(v, (dict, list, tuple)) else v)
    out.to_csv(OUT / filename, index=index)

tables = {'outer_loso_results.csv': outer_results,
          'outer_classifier_results.csv': model_outer_results,
          'outer_feature_set_classifier_results.csv': pair_outer_results,
          'primary_outer_summary.csv': primary_summary,
          'outer_classifier_summary.csv': model_summary,
          'outer_feature_set_classifier_summary.csv': pair_summary,
          'inner_cv_best_params_per_feature_set.csv': inner_results,
          'inner_cv_all_candidates.csv': inner_all_results,
          'inner_feature_set_performance.csv': feature_set_summary,
          'best_feature_set_summary_descriptive_only.csv': best_feature_summary,
          'outer_predictions.csv': outer_predictions,
          'outer_classifier_predictions.csv': model_outer_predictions,
          'outer_feature_set_classifier_predictions.csv': pair_outer_predictions,
          'selected_features_by_outer_fold.csv': selected_feature_table,
          'fit_warnings.csv': warning_log,
          'final_training_warnings.csv': pd.DataFrame(final_training_warnings, columns=WARNING_COLUMNS)}
if TRAIN_FINAL_MODEL:
    tables['final_all_participants_cv_best_params_per_feature_set.csv'] = final_search_results
    tables['final_all_participants_cv_all_candidates.csv'] = final_search_all_results
for name, frame in tables.items():
    export_table(frame, name)
export_table(feature_frequency, 'selected_feature_frequency.csv', index=True)
export_table(report_df, 'pooled_classification_report.csv', index=True)
for model_name, report in model_reports.items():
    export_table(report, f'classification_report_{slug(model_name)}.csv', index=True)

atomic_json({'run_id': RUN_ID, 'run_mode': RUN_MODE, 'nested_evaluation_complete': True,
             'final_model_exported': final_bundle is not None, 'fit_counts_this_execution': FIT_COUNTS,
             'primary_results': primary_summary.to_dict('records')}, OUT / 'run_summary.json')

readme = f"""SmartGlove corrected classification — {RUN_MODE.upper()} run
Run ID: {RUN_ID}
Dataset SHA-256: {DATASET_SHA256}

primary_outer_summary.csv: primary generalization estimate of the full selection procedure.
outer_classifier_summary.csv: each classifier tuned separately inside the outer development set.
outer_feature_set_classifier_summary.csv: fixed F1-F6 feature-set/classifier comparisons.
inner_feature_set_performance.csv: INNER-CV descriptive feature-set results, not independent test accuracies.
fit_warnings.csv: review any convergence/numerical warnings.
calibration_reference_diagnostics.csv: measured references; span is diagnostic only in the primary experiment.
final_model_bundle.joblib: all-participant refit, if enabled; no new independent test score.
smartglove_inference.py: Python inference from 50 sensor rows with the required session calibration.
inference_requirements.txt: package versions used to create the model.

SD is between-participant variation, not a confidence interval.
F2-F6 require the current-session neutral calibration; F1 uses raw flex medians.
Do not choose a new model from outer scores and reuse its best score as an untouched evaluation.
SMOKE outputs, when present, are software checks and are not the full experiment.
Checkpoints can resume matching runs only while these files remain accessible.
"""
(OUT / 'RESULTS_README.txt').write_text(readme, encoding='utf-8')
archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT.parent, base_dir=OUT.name)
print('Saved results:', OUT.resolve())
print('Downloadable archive:', archive)
try:
    from IPython.display import FileLink
    display(FileLink(archive))
except ImportError:
    pass
try:
    from google.colab import files
    print('To download in Colab, run: files.download(archive)')
except ImportError:
    pass
# In Colab, uncomment the next line when you want to download the ZIP:
# files.download(archive)


## What to report and verify

- State **500 trial-level examples from 10 participants**, 50 samples per trial.
- State that primary feature analysis used the six fixed **F1-F6** predictor blocks.
- State that outer evaluation used participant LOSO and inner tuning used participant
  LOSO on the remaining development participants.
- State that **mean participant macro-F1** was the primary inner selection metric.
- Report held-out accuracy, macro-F1, participant results and confusion matrices.
- Span normalization is secondary/diagnostic and is not part of the primary search.
- Check warnings before claiming every fit converged.
- Ten participants provide limited evidence about a broader user population.

**Validation of this deliverable:** the revised notebook is syntax-checked and its
data loading, exact-count assertions, calibration mapping, F1-F6 feature generation,
inner participant-LOSO splitting, model grids and a reduced real-data execution path
are rechecked before delivery.
